# 函数调用：交互式实验

本 notebook 把中文镜像站中的 [Cookbook](/cookbooks/function_calling/) 改写成可以逐格运行、修改输入并观察结果的最小实验。
用 Choice 选择注册函数，再由代码执行分派。

运行方式与 `../03_架构模式/01_架构模式.ipynb` 一致：有有效的 `TYPESAFE_API_KEY` 时调用真实的
TypeSafe API；没有 Key 或返回 401 时使用内置的离线示例答案。后续代码不区分两种模式，便于先学习
控制流，再切换到真实模型观察概率和置信度。

> 学习提示：先顺序运行全部单元格，再回到“定义 state”或“定义问题”的单元格修改内容，重新运行后面的单元格。
> API Key 只从环境变量读取，不能写进 notebook。


## 0. 准备

### 0.1 安装依赖

In [ ]:
%pip install -q -U typesafe-sdk

### 0.2 创建客户端

In [ ]:
import os
import statistics
import time
from pprint import pprint

from typesafe_sdk import (
    Choice,
    Score,
    Noul,
    TypeSafeClient,
    TypeSafeAuthenticationError,
)

API_KEY = os.environ.get("TYPESAFE_API_KEY", "")
client = TypeSafeClient(api_key=API_KEY, model="jev-latest") if API_KEY else None
print("客户端已创建：模型=jev-latest，Key=", "已配置" if API_KEY else "未配置（将使用离线示例）")


### 0.3 离线响应与统一调用入口

In [ ]:
class _FakeAnswer:
    def __init__(self, type_, **values):
        self.type = type_
        for key, value in values.items():
            setattr(self, key, value)


class _FakeResponse:
    def __init__(self, answers):
        self.answers = answers
        self.nouls = {k: v for k, v in answers.items() if v.type == "noul"}
        self.choices = {k: v for k, v in answers.items() if v.type == "choice"}
        self.scores = {k: v for k, v in answers.items() if v.type == "score"}
        self.model = "jev-latest（离线示例）"
        self.usage = _FakeAnswer("usage", input_tokens=0, output_tokens=0)


class TS:
    offline = False
    _warned = False

    @classmethod
    def call(cls, state, questions, offline_answers):
        if client is None:
            cls.offline = True
            if not cls._warned:
                cls._warned = True
                print("⚠️ 未设置有效 TYPESAFE_API_KEY，以下输出使用内置离线示例。")
            return _FakeResponse(offline_answers)
        try:
            return client.system_one(state, questions)
        except TypeSafeAuthenticationError:
            cls.offline = True
            if not cls._warned:
                cls._warned = True
                print("⚠️ 未设置有效 TYPESAFE_API_KEY，以下输出使用内置离线示例。")
            return _FakeResponse(offline_answers)


def answer_line(name, answer):
    if answer.type == "noul":
        return f"{name}: noul={answer.noul:.2f}"
    if answer.type == "choice":
        return f"{name}: choice={answer.choice} confidence={answer.confidence:.2f}"
    return f"{name}: score={answer.score:.2f} confidence={answer.confidence:.2f}"


print("模式：", "离线示例" if TS.offline else "真实 API（首次调用后确定）")


### 0.4 连通性测试

In [ ]:
if client is None:
    TS.offline = True
    print("⚠️ API Key 未设置，后续单元格使用离线示例。")
else:
    try:
        ping = client.system_one("你好", {"is_greeting": Noul(instructions="这段文字是在打招呼吗？")})
        print("✅ API 连通正常，后续单元格会使用真实结果。")
    except TypeSafeAuthenticationError:
        TS.offline = True
        print("⚠️ API Key 无效，后续单元格使用离线示例。")


## 1. 函数调用

先用 Choice 选择确定的函数，再由代码校验参数并调用本地函数。模型不会直接执行副作用。


### 1.1 定义用户请求和函数目录

In [ ]:
REQUEST = "请查询订单 TS-2048 的物流状态，并告诉我预计送达日期。"
FUNCTIONS = {
    "lookup_order": lambda order_id: f"订单 {order_id}：运输中，预计周五送达。",
    "refund_order": lambda order_id: f"订单 {order_id}：退款申请已创建。",
    "update_address": lambda order_id: f"订单 {order_id}：地址修改需要人工确认。",
}
QUESTIONS = {
    "function": Choice(
        instructions="用户最想执行哪个函数？",
        criteria={
            "lookup_order": "查询订单物流或状态",
            "refund_order": "申请订单退款",
            "update_address": "修改订单收货地址",
        },
    ),
    "needs_confirmation": Noul(instructions="执行这个函数前是否需要用户再次确认？"),
}
print("函数目录和问题已定义：函数数=", len(FUNCTIONS), "，问题数=", len(QUESTIONS))


### 1.2 调用并让代码完成分派

In [ ]:
OFFLINE = {
    "function": _FakeAnswer("choice", choice="lookup_order", confidence=.96,
                             probabilities={"lookup_order": .96}),
    "needs_confirmation": _FakeAnswer("noul", noul=.08),
}
response = TS.call(REQUEST, QUESTIONS, OFFLINE)
for name, answer in response.answers.items():
    print(answer_line(name, answer))

selected = response.choices["function"].choice
if selected not in FUNCTIONS:
    raise ValueError(f"模型返回了未注册函数：{selected}")
if response.nouls["needs_confirmation"].noul >= 0.50:
    print("→ 先请求用户确认，不执行副作用。")
else:
    print("→ 确定性代码执行：", FUNCTIONS[selected]("TS-2048"))


观察：Choice 只决定“调用哪个已注册函数”，函数本身、参数校验和副作用都留在 Python 代码中。

## 知识补充
- **选择与执行分离**：模型只从注册清单里选（Choice），参数解析、权限校验、真正执行全在代码——比 LLM function calling 更可控，也不会出现模型编造参数。
- **边界**：Jev 不生成参数；参数需要从 state 里用代码抽取（正则/解析器，见 `14_日期抽取.ipynb` 和 `15_预解析值抽取.ipynb`），模糊表述才交给语义。
- **进阶阅读**：先判断"要不要调工具"再选工具，是 `../03_架构模式/01_架构模式.ipynb` 里意图路由的标准前缀。

## 小结

这本 notebook 的边界很清楚：TypeSafe 只负责受限、可编程的判断；排序、阈值、分组、重建文本和
函数分派都由 Python 代码完成。修改输入或问题后重新运行，就能观察“模型答案 → 确定性代码”的变化。
